In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [32]:
train = pd.read_csv("Training Dataset.csv")
test = pd.read_csv("Test Dataset.csv")

In [35]:
train.shape
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [36]:
test.shape
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            367 non-null    object 
 1   Gender             356 non-null    object 
 2   Married            367 non-null    object 
 3   Dependents         357 non-null    object 
 4   Education          367 non-null    object 
 5   Self_Employed      344 non-null    object 
 6   ApplicantIncome    367 non-null    int64  
 7   CoapplicantIncome  367 non-null    int64  
 8   LoanAmount         362 non-null    float64
 9   Loan_Amount_Term   361 non-null    float64
 10  Credit_History     338 non-null    float64
 11  Property_Area      367 non-null    object 
dtypes: float64(3), int64(2), object(7)
memory usage: 34.5+ KB


In [41]:
missing_values_train=train.isnull().sum()
print("Missing values in train dataset:\n", missing_values_train[missing_values_train > 0])

Missing values in train dataset:
 Gender              13
Married              3
Dependents          15
Self_Employed       32
LoanAmount          22
Loan_Amount_Term    14
Credit_History      50
dtype: int64


In [39]:
missing_values_test = test.isnull().sum()
print("Missing values in test dataset:\n", missing_values_test[missing_values_test > 0])

Missing values in test dataset:
 Gender              11
Dependents          10
Self_Employed       23
LoanAmount           5
Loan_Amount_Term     6
Credit_History      29
dtype: int64


In [57]:
num_features = train.select_dtypes(include=[np.number]).columns
num_features = num_features.drop('Loan_ID', errors='ignore')


In [58]:
cat_features = train.select_dtypes(include=[object]).columns
cat_features = cat_features.drop('Loan_ID', errors='ignore')

In [62]:
for feature in train.columns:
    if train[feature].dtype in ['int64', 'float64']: 
        train[feature] = train[feature].fillna(train[feature].median())
    if feature in test.columns and test[feature].dtype in ['int64', 'float64']:  
        test[feature] = test[feature].fillna(test[feature].median())

In [63]:
for feature in cat_features:
    train[feature] = train[feature].fillna(train[feature].mode()[0])
    if feature in test.columns:
        test[feature] = test[feature].fillna(test[feature].mode()[0])

In [64]:
combined = pd.concat([train, test], axis=0)


In [65]:
combined = pd.get_dummies(combined, columns=cat_features)

In [66]:
train = combined[:train.shape[0]]
test = combined[train.shape[0]:]

In [69]:
train.loc[:, 'LoanAmount_log'] = np.log(train['LoanAmount'] + 1)
test.loc[:, 'LoanAmount_log'] = np.log(test['LoanAmount'] + 1)

In [71]:
train.loc[:, 'TotalIncome'] = train['ApplicantIncome'] + train['CoapplicantIncome']
test.loc[:, 'TotalIncome'] = test['ApplicantIncome'] + test['CoapplicantIncome']

In [73]:
train.loc[:, 'TotalIncome_log'] = np.log(train['TotalIncome'] + 1)
test.loc[:, 'TotalIncome_log'] = np.log(test['TotalIncome'] + 1)

In [74]:
scaler = StandardScaler()
num_features = train.select_dtypes(include=[np.number]).columns
num_features = num_features.drop(['Loan_ID'], errors='ignore')

In [76]:
train.loc[:, num_features] = scaler.fit_transform(train[num_features])
test.loc[:, num_features] = scaler.transform(test[num_features])
